# Data cleaning

In [114]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport


## 1. Importing data

In [115]:
csv_path = "../data/raw/store-data-6aa6d7a3f171f140353680.csv"

df = pd.read_csv(csv_path)

## 2. Fixing text

In [116]:
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

df["Segment"] = df["Segment"].replace({"Consumerr": "Consumer"})
df["Segment"] = df["Segment"].replace({"Home Ofice": "Home Office"})
df["Segment"] = df["Segment"].replace({"Corporrate": "Corporate"})


df["Customer Name"] = df["Customer Name"].str.strip().str.title()
df["Category"] = df["Category"].str.strip().str.title()
df["Segment"] = df["Segment"].str.strip().str.title()
df["State"] = df["State"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()


## 3.Removing duplicates

In [117]:
df = df.drop_duplicates()

## 4. Fixing invalid dates

In [118]:
for column in ['Order Date', 'Ship Date']:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')

if {'Order Date', 'Ship Date'}.issubset(df.columns):
    invalid_dates = df[
        df['Order Date'].notna() &
        df['Ship Date'].notna() &
        (df['Ship Date'] < df['Order Date'])
    ]
    df = df.drop(invalid_dates.index)

## 5. Filling missing data

In [119]:
df['Customer Name'] = df['Customer Name'].fillna(
    df.groupby('Customer ID')['Customer Name'].transform('first')
)

df['Postal Code'] = df['Postal Code'].fillna(
    df.groupby('City')['Postal Code'].transform('first')
)


df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df = df.dropna(subset = ['Shipping Days'])


median_days = df.groupby("Ship Mode")["Shipping Days"].median()
median_days = pd.to_timedelta(df["Ship Mode"].map(median_days))

df["Ship Date"] = np.where(df["Shipping Days"] > 90, df["Order Date"] + median_days, df["Ship Date"])


median_days = df.groupby("Ship Mode")["Shipping Days"].median()

def find_mode(days):
    difference = (median_days - days).abs()
    return difference.idxmin()

missing = df["Ship Mode"].isna()

df.loc[missing, "Ship Mode"] = df.loc[missing, "Shipping Days"].apply(find_mode)

df["Quantity"] = df['Quantity'].fillna(df['Quantity'].mode().iloc[0])
df = df.dropna(subset = ['Sales'])

df["Discount"] = df["Discount"].fillna(df.groupby(["Product ID", "Sales"])["Discount"].transform(lambda x: x.median()))
df = df[~((df['Discount'] < 0) | (df['Discount'] > 1))]

df["Quantity"] = df["Quantity"].fillna(
    df.groupby(["Product ID", "Sales"])["Quantity"].transform(lambda x: x.median())
)

df["Quantity"] = df["Quantity"].round().astype(int)

df["Sales"] = df["Sales"].fillna(df.groupby(["Product ID", "Quantity"])["Sales"].transform(lambda x: x.median()))


# display(df["Discount"])
# display(df[df["Quantity"] < 0])

missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100)
})
display(missing.sort_values('missing_count', ascending=False))




,missing_count,missing_percentage
Customer Name,1,0.010641
Row ID,0,0.000000
Order Date,0,0.000000
Ship Date,0,0.000000
Ship Mode,0,0.000000
Order ID,0,0.000000
Customer ID,0,0.000000
Segment,0,0.000000
Country,0,0.000000
City,0,0.000000


## 5. Removing irrelevant data

In [120]:
df = df.drop_duplicates()
df = df.dropna(subset=['Customer Name'])
df = df[df['Quantity'] > 0]

missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100)
})
display(missing.sort_values('missing_count', ascending=False))



,missing_count,missing_percentage
Row ID,0,0.0
Order ID,0,0.0
Order Date,0,0.0
Ship Date,0,0.0
Ship Mode,0,0.0
Customer ID,0,0.0
Customer Name,0,0.0
Segment,0,0.0
Country,0,0.0
City,0,0.0


In [121]:

profile = ProfileReport(df)
profile.to_file("../reports/report_cleaning.html")



Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 22/22 [00:00<00:00, 76.36it/s] 


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]